# SINet-V2 + PVT-v2 — Training on Kaggle

**Trước khi chạy:**
1. GPU: `Session → Accelerator → GPU T4 x2` hoặc `P100`
2. Internet: `Session → Internet → On`
3. Thêm dataset `sinet-v2-data` (chứa train/test zip) vào `/kaggle/input/`

## Bước 1 — Cài thư viện

In [1]:
!pip install thop tensorboardX timm imageio gdown -q
print('✅ Cài xong!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

## Bước 2 — Kiểm tra GPU

In [2]:
import os, torch
print('=== GPU ===')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('\n=== Input datasets ===')
!ls /kaggle/input/

=== GPU ===
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
CUDA: True
GPU: Tesla T4

=== Input datasets ===
datasets


## Bước 3 — Clone SINet-V2

In [3]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('SINet-V2'):
    !git clone https://github.com/GewelsJI/SINet-V2.git
    print('✅ Clone xong!')
else:
    print('✅ Repo đã có.')
os.chdir('/kaggle/working/SINet-V2')
os.makedirs('./lib', exist_ok=True)
print('Thư mục hiện tại:', os.getcwd())

Cloning into 'SINet-V2'...
remote: Enumerating objects: 509, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 509 (delta 178), reused 189 (delta 128), pack-reused 265 (from 1)
Receiving objects: 100% (509/509), 1.82 MiB | 12.07 MiB/s, done.
Resolving deltas: 100% (308/308), done.
✅ Clone xong!
Thư mục hiện tại: /kaggle/working/SINet-V2


## Bước 4 — Ghi `lib/Network_PVTv2_GRA_NCD.py`

Thay backbone Res2Net50 → PVT-v2-B2. Giữ nguyên RFB, NCD, GRA.

In [4]:
import os
os.chdir('/kaggle/working/SINet-V2')
os.makedirs('./lib', exist_ok=True)

network_code = '''
# Network_PVTv2_GRA_NCD.py
# SINet-V2 voi PVT-v2 backbone thay Res2Net50
# Thay doi:
#   - Backbone: res2net50 -> pvt_v2_b2
#   - RFB input channels: 512/1024/2048 -> 128/320/512
#   - forward(): backbone(x) tra ve list 4 stage

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from lib.pvt_v2 import pvt_v2_b2
    _USE_LOCAL = True
except ImportError:
    _USE_LOCAL = False

if not _USE_LOCAL:
    try:
        import timm
        _USE_TIMM = True
    except ImportError:
        raise ImportError("Can not find PVT-v2. Install timm: pip install timm")


class PVTv2_Timm(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.model = timm.create_model(
            'pvt_v2_b2', pretrained=pretrained, features_only=True)
    def forward(self, x):
        return self.model(x)


class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size,
                 stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size,
                              stride=stride, padding=padding,
                              dilation=dilation, bias=False)
        self.bn   = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class RFB_modified(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(BasicConv2d(in_channel, out_channel, 1))
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, (1,3), padding=(0,1)),
            BasicConv2d(out_channel, out_channel, (3,1), padding=(1,0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3))
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, (1,5), padding=(0,2)),
            BasicConv2d(out_channel, out_channel, (5,1), padding=(2,0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5))
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, (1,7), padding=(0,3)),
            BasicConv2d(out_channel, out_channel, (7,1), padding=(3,0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7))
        self.conv_cat = BasicConv2d(4*out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)
    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        return self.relu(self.conv_cat(torch.cat((x0,x1,x2,x3),1)) + self.conv_res(x))


class NeighborConnectionDecoder(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.upsample       = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2*channel, 2*channel, 3, padding=1)
        self.conv_concat2   = BasicConv2d(2*channel, 2*channel, 3, padding=1)
        self.conv_concat3   = BasicConv2d(3*channel, 3*channel, 3, padding=1)
        self.conv4          = BasicConv2d(3*channel, 3*channel, 3, padding=1)
        self.conv5          = nn.Conv2d(3*channel, 1, 1)
    def forward(self, x1, x2, x3):
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = (self.conv_upsample2(self.upsample(x2_1))
                * self.conv_upsample3(self.upsample(x2)) * x3)
        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)
        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)
        x3_2 = self.conv_concat3(x3_2)
        return self.conv5(self.conv4(x3_2))


class GRA(nn.Module):
    def __init__(self, channel, subchannel):
        super().__init__()
        self.group = channel // subchannel
        self.conv  = nn.Sequential(
            nn.Conv2d(channel + self.group, channel, 3, padding=1),
            nn.ReLU(True))
        self.score = nn.Conv2d(channel, 1, 3, padding=1)
    def forward(self, x, y):
        g = self.group
        if g == 1:
            x_cat = torch.cat((x, y), 1)
        else:
            xs = torch.chunk(x, g, dim=1)
            parts = []
            for xi in xs:
                parts += [xi, y]
            x_cat = torch.cat(parts, 1)
        x = x + self.conv(x_cat)
        y = y + self.score(x)
        return x, y


class ReverseStage(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.weak_gra   = GRA(channel, channel)
        self.medium_gra = GRA(channel, 8)
        self.strong_gra = GRA(channel, 1)
    def forward(self, x, y):
        y = -1 * torch.sigmoid(y) + 1
        x, y = self.weak_gra(x, y)
        x, y = self.medium_gra(x, y)
        _, y = self.strong_gra(x, y)
        return y


PVT_CHANNELS = {
    'b0': [32,  64,  160, 256],
    'b1': [64,  128, 320, 512],
    'b2': [64,  128, 320, 512],
    'b3': [64,  128, 320, 512],
    'b4': [64,  128, 320, 512],
    'b5': [64,  128, 320, 512],
}


class Network(nn.Module):
    def __init__(self, channel=32, pvt_variant='b2', imagenet_pretrained=True):
        super().__init__()
        if _USE_LOCAL:
            fn_map = {'b2': pvt_v2_b2}
            self.backbone = fn_map[pvt_variant](pretrained=imagenet_pretrained)
        else:
            self.backbone = PVTv2_Timm(pretrained=imagenet_pretrained)

        ch = PVT_CHANNELS[pvt_variant]
        self.rfb2_1 = RFB_modified(ch[1], channel)
        self.rfb3_1 = RFB_modified(ch[2], channel)
        self.rfb4_1 = RFB_modified(ch[3], channel)
        self.NCD    = NeighborConnectionDecoder(channel)
        self.RS5    = ReverseStage(channel)
        self.RS4    = ReverseStage(channel)
        self.RS3    = ReverseStage(channel)

    def forward(self, x):
        feats = self.backbone(x)
        x2, x3, x4 = feats[1], feats[2], feats[3]
        x2_rfb = self.rfb2_1(x2)
        x3_rfb = self.rfb3_1(x3)
        x4_rfb = self.rfb4_1(x4)

        S_g      = self.NCD(x4_rfb, x3_rfb, x2_rfb)
        S_g_pred = F.interpolate(S_g, scale_factor=8,
                                 mode='bilinear', align_corners=False)

        g5 = F.interpolate(S_g, scale_factor=0.25,
                           mode='bilinear', align_corners=False)
        ra4 = self.RS5(x4_rfb, g5)
        S5  = ra4 + g5
        S5_pred = F.interpolate(S5, scale_factor=32,
                                mode='bilinear', align_corners=False)

        g4 = F.interpolate(S5, scale_factor=2,
                           mode='bilinear', align_corners=False)
        ra3 = self.RS4(x3_rfb, g4)
        S4  = ra3 + g4
        S4_pred = F.interpolate(S4, scale_factor=16,
                                mode='bilinear', align_corners=False)

        g3 = F.interpolate(S4, scale_factor=2,
                           mode='bilinear', align_corners=False)
        ra2 = self.RS3(x2_rfb, g3)
        S3  = ra2 + g3
        S3_pred = F.interpolate(S3, scale_factor=8,
                                mode='bilinear', align_corners=False)

        return S_g_pred, S5_pred, S4_pred, S3_pred
'''

with open('./lib/Network_PVTv2_GRA_NCD.py', 'w') as f:
    f.write(network_code.strip())

print(f'✅ Ghi xong lib/Network_PVTv2_GRA_NCD.py  '
      f'({os.path.getsize("./lib/Network_PVTv2_GRA_NCD.py"):,} bytes)')

✅ Ghi xong lib/Network_PVTv2_GRA_NCD.py  (7,686 bytes)


## Bước 5 — Ghi `MyTrain_Val.py`

In [5]:
import os
os.chdir('/kaggle/working/SINet-V2')

train_code = '''
# MyTrain_Val.py — SINet-V2 with PVT-v2 backbone
# Changes vs original: cosine LR + warmup, differential LR, CSV log

import shutil, os, csv
import torch
import torch.nn.functional as F
import numpy as np
from datetime import datetime
from torchvision.utils import make_grid
from lib.Network_PVTv2_GRA_NCD import Network
from utils.data_val import get_loader, test_dataset
from utils.utils import clip_gradient
from tensorboardX import SummaryWriter
import logging
import torch.backends.cudnn as cudnn

best_mae   = 1.0
best_epoch = 0
step       = 0
CSV_LOG    = None


def structure_loss(pred, mask):
    weit  = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)
    wbce  = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")
    wbce  = (weit * wbce).sum(dim=(2,3)) / weit.sum(dim=(2,3))
    pred  = torch.sigmoid(pred)
    inter = ((pred * mask) * weit).sum(dim=(2,3))
    union = ((pred + mask) * weit).sum(dim=(2,3))
    wiou  = 1 - (inter + 1) / (union - inter + 1)
    return (wbce + wiou).mean()


def cosine_lr_with_warmup(optimizer, base_lr, epoch, total_epoch,
                          warmup_epoch=3, lr_min=1e-6):
    if epoch <= warmup_epoch:
        lr = base_lr * 0.1 + (base_lr - base_lr*0.1) * (epoch / warmup_epoch)
    else:
        progress = (epoch - warmup_epoch) / max(total_epoch - warmup_epoch, 1)
        lr = lr_min + 0.5 * (base_lr - lr_min) * (1 + np.cos(np.pi * progress))
    for pg in optimizer.param_groups:
        pg["lr"] = lr * 0.1 if pg.get("is_backbone", False) else lr
    return lr


def init_csv(path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow(
            ["epoch","lr","loss_total","loss_init","loss_final",
             "mae","best_mae","best_epoch"])


def log_csv(path, epoch, lr, lt, li, lf, mae, best_mae, best_epoch):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow(
            [epoch, f"{lr:.2e}", f"{lt:.4f}", f"{li:.4f}",
             f"{lf:.4f}", f"{mae:.6f}", f"{best_mae:.6f}", best_epoch])


def train(train_loader, model, optimizer, epoch, save_path, writer):
    global step
    model.train()
    loss_all = loss_init_all = loss_final_all = 0
    n = 0
    try:
        for i, (images, gts) in enumerate(train_loader, start=1):
            optimizer.zero_grad()
            images, gts = images.cuda(), gts.cuda()
            p = model(images)
            loss_init  = (structure_loss(p[0], gts)
                          + structure_loss(p[1], gts)
                          + structure_loss(p[2], gts))
            loss_final = structure_loss(p[3], gts)
            loss = loss_init + loss_final
            loss.backward()
            clip_gradient(optimizer, opt.clip)
            optimizer.step()
            step += 1; n += 1
            loss_all       += loss.item()
            loss_init_all  += loss_init.item()
            loss_final_all += loss_final.item()
            if i % 50 == 0 or i == total_step or i == 1:
                msg = (f"{datetime.now()} Ep[{epoch:03d}/{opt.epoch}] "
                       f"Step[{i:04d}/{total_step}] "
                       f"Total:{loss.item():.4f} "
                       f"Init:{loss_init.item():.4f} "
                       f"Final:{loss_final.item():.4f}")
                print(msg)
                logging.info(msg)
                writer.add_scalars("Loss", {
                    "init":  loss_init.item(),
                    "final": loss_final.item(),
                    "total": loss.item()}, global_step=step)
        avg_lt = loss_all / n
        avg_li = loss_init_all / n
        avg_lf = loss_final_all / n
        logging.info(f"[Train] Ep[{epoch}/{opt.epoch}] AVG Loss:{avg_lt:.4f}")
        writer.add_scalar("Loss-epoch", avg_lt, global_step=epoch)
        if epoch % 5 == 0:
            p = save_path + f"Net_epoch_{epoch}.pth"
            torch.save(model.state_dict(), p)
            print(f"[Saved] {p}")
        return avg_lt, avg_li, avg_lf
    except KeyboardInterrupt:
        torch.save(model.state_dict(), save_path + f"Net_epoch_{epoch}_interrupt.pth")
        raise


def val(test_loader, model, epoch, save_path, writer):
    global best_mae, best_epoch
    model.eval()
    mae_sum = 0
    with torch.no_grad():
        for _ in range(test_loader.size):
            image, gt, name, _ = test_loader.load_data()
            gt = np.asarray(gt, np.float32)
            gt /= (gt.max() + 1e-8)
            res = model(image.cuda())
            res = F.interpolate(res[3], size=gt.shape,
                                mode="bilinear", align_corners=False)
            res = res.sigmoid().cpu().numpy().squeeze()
            res = (res - res.min()) / (res.max() - res.min() + 1e-8)
            mae_sum += np.abs(res - gt).sum() / (gt.shape[0]*gt.shape[1])
    mae = mae_sum / test_loader.size
    writer.add_scalar("MAE", torch.tensor(mae), global_step=epoch)
    print(f"Ep:{epoch}  MAE:{mae:.6f}  bestMAE:{best_mae:.6f}  bestEp:{best_epoch}")
    if epoch == 1 or mae < best_mae:
        best_mae = mae; best_epoch = epoch
        p = save_path + "Net_epoch_best.pth"
        torch.save(model.state_dict(), p)
        print(f"  -> New best! Saved epoch {epoch}")
    logging.info(f"[Val] Ep:{epoch} MAE:{mae:.6f} best:{best_mae:.6f} ep:{best_epoch}")
    return mae


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--epoch",        type=int,   default=25)
    parser.add_argument("--lr",           type=float, default=5e-4)
    parser.add_argument("--batchsize",    type=int,   default=36)
    parser.add_argument("--trainsize",    type=int,   default=352)
    parser.add_argument("--clip",         type=float, default=0.5)
    parser.add_argument("--warmup_epoch", type=int,   default=3)
    parser.add_argument("--lr_min",       type=float, default=1e-6)
    parser.add_argument("--load",         type=str,   default=None)
    parser.add_argument("--gpu_id",       type=str,   default="0")
    parser.add_argument("--pvt_variant",  type=str,   default="b2")
    parser.add_argument("--train_root",   type=str,
                        default="./Dataset/TrainValDataset/")
    parser.add_argument("--val_root",     type=str,
                        default="./Dataset/TestDataset/CAMO/")
    parser.add_argument("--save_path",    type=str,
                        default="./snapshot/SINet_V2_PVTv2/")
    opt = parser.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = opt.gpu_id
    cudnn.benchmark = True

    os.makedirs(opt.save_path, exist_ok=True)
    logging.basicConfig(
        filename=opt.save_path + "log.log",
        format="[%(asctime)s-%(filename)s-%(levelname)s:%(message)s]",
        level=logging.INFO, filemode="a", datefmt="%Y-%m-%d %I:%M:%S %p")
    logging.info(">>> Config: epoch=%d lr=%e batch=%d size=%d" % (
        opt.epoch, opt.lr, opt.batchsize, opt.trainsize))

    CSV_LOG = opt.save_path + "training_log.csv"
    init_csv(CSV_LOG)

    model = Network(channel=32, pvt_variant=opt.pvt_variant,
                    imagenet_pretrained=True).cuda()
    if opt.load:
        model.load_state_dict(torch.load(opt.load)); print(f"Loaded: {opt.load}")

    backbone_params = list(model.backbone.parameters())
    head_params     = [p for p in model.parameters()
                       if not any(p is q for q in backbone_params)]
    optimizer = torch.optim.Adam([
        {"params": backbone_params, "lr": opt.lr*0.1, "is_backbone": True},
        {"params": head_params,     "lr": opt.lr}], lr=opt.lr)

    train_loader = get_loader(
        opt.train_root + "Imgs/", opt.train_root + "GT/",
        batchsize=opt.batchsize, trainsize=opt.trainsize, num_workers=4)
    test_loader  = test_dataset(
        opt.val_root + "Imgs/", opt.val_root + "GT/", opt.trainsize)
    total_step   = len(train_loader)

    writer = SummaryWriter(opt.save_path + "summary")
    print(f"Training steps/epoch: {total_step}")

    for epoch in range(1, opt.epoch + 1):
        cur_lr = cosine_lr_with_warmup(
            optimizer, opt.lr, epoch, opt.epoch,
            opt.warmup_epoch, opt.lr_min)
        lt, li, lf = train(train_loader, model, optimizer,
                           epoch, opt.save_path, writer)
        mae = val(test_loader, model, epoch, opt.save_path, writer)
        log_csv(CSV_LOG, epoch, cur_lr, lt, li, lf, mae, best_mae, best_epoch)
        print(f"[Ep {epoch:3d}/{opt.epoch}] LR={cur_lr:.2e} "
              f"Loss={lt:.4f} MAE={mae:.6f}")
    writer.close()
    print("Training done!")
'''

with open('./MyTrain_Val.py', 'w') as f:
    f.write(train_code.strip())

sz = os.path.getsize('./MyTrain_Val.py')
print(f'✅ Ghi xong MyTrain_Val.py  ({sz:,} bytes)')

✅ Ghi xong MyTrain_Val.py  (8,626 bytes)


## Bước 6 — Chuẩn bị dataset

Giải nén từ `/kaggle/input/sinet-v2-data/` hoặc tự tải.

In [6]:
import shutil, os
os.chdir('/kaggle/working/SINet-V2')

INPUT_BASE = '/kaggle/input/datasets/tanleluv/sinet-v2-dataset/sinet_v2_dataset'
TRAIN_SRC  = f'{INPUT_BASE}/COD-TrainDataset'
TEST_SRC   = f'{INPUT_BASE}/COD-TestDataset'

TRAIN_DIR  = './Dataset/TrainValDataset'
TEST_DIR   = './Dataset/TestDataset'

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR,  exist_ok=True)

# Copy train
if not os.path.exists(f'{TRAIN_DIR}/Imgs'):
    print('⏳ Copy train dataset...')
    os.system(f'cp -r {TRAIN_SRC}/. {TRAIN_DIR}/')
    print('✅ Train dataset sẵn sàng!')
else:
    print('✅ Train dataset đã có.')

# Copy test
if not os.path.exists(f'{TEST_DIR}/CAMO'):
    print('⏳ Copy test dataset...')
    os.system(f'cp -r {TEST_SRC}/. {TEST_DIR}/')
    print('✅ Test dataset sẵn sàng!')
else:
    print('✅ Test dataset đã có.')

print('\n=== Cấu trúc Dataset ===')
for root, dirs, files in os.walk('./Dataset'):
    depth = root.replace('./Dataset', '').count(os.sep)
    if depth > 2: continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 2:
        print(f'{indent}  ({len(files)} files)')

⏳ Copy train dataset...
✅ Train dataset sẵn sàng!
⏳ Copy test dataset...
✅ Test dataset sẵn sàng!

=== Cấu trúc Dataset ===
Dataset/
  TrainValDataset/
    Imgs/
      (4040 files)
    GT/
      (4040 files)
  TestDataset/
    COD10K/
      (0 files)
    CHAMELEON/
      (0 files)
    CAMO/
      (0 files)
    NC4K/
      (0 files)


## Bước 7 — Kiểm tra model (sanity check)

In [7]:
import sys, torch
sys.path.insert(0, '/kaggle/working/SINet-V2')
os.chdir('/kaggle/working/SINet-V2')

from lib.Network_PVTv2_GRA_NCD import Network

model = Network(channel=32, pvt_variant='b2', imagenet_pretrained=True).cuda()
dummy = torch.randn(2, 3, 352, 352).cuda()
with torch.no_grad():
    outs = model(dummy)
print('Output shapes:')
for i, o in enumerate(outs):
    print(f'  pred[{i}]: {tuple(o.shape)}')

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params : {total/1e6:.2f}M')
print(f'Trainable    : {train/1e6:.2f}M')
del model, dummy
torch.cuda.empty_cache()
print('✅ Model OK!')

model.safetensors:   0%|          | 0.00/101M [00:00<?, ?B/s]

Output shapes:
  pred[0]: (2, 1, 352, 352)
  pred[1]: (2, 1, 352, 352)
  pred[2]: (2, 1, 352, 352)
  pred[3]: (2, 1, 352, 352)

Total params : 25.69M
Trainable    : 25.69M
✅ Model OK!


## Bước 8 — Train 70 epoch

> Checkpoint lưu mỗi 5 epoch tại `snapshot/SINet_V2_PVTv2/`

In [8]:
import time, os
os.chdir('/kaggle/working/SINet-V2')
os.makedirs('./snapshot/SINet_V2_PVTv2', exist_ok=True)

print('🚀 Bắt đầu train...')
start = time.time()

!python MyTrain_Val.py \
    --epoch 70 \
    --lr 5e-4 \
    --batchsize 32 \
    --trainsize 352 \
    --clip 0.5 \
    --warmup_epoch 3 \
    --lr_min 1e-6 \
    --pvt_variant b2 \
    --gpu_id 0 \
    --save_path ./snapshot/SINet_V2_PVTv2/ \
    --train_root ./Dataset/TrainValDataset/ \
    --val_root   ./Dataset/TestDataset/CAMO/

elapsed = (time.time() - start) / 3600
print(f'\n⏱️  Tổng thời gian: {elapsed:.2f} giờ')

🚀 Bắt đầu train...
Training steps/epoch: 127
2026-06-08 16:12:14.743088 Ep[001/70] Step[0001/127] Total:6.3883 Init:4.6466 Final:1.7417
2026-06-08 16:14:11.657949 Ep[001/70] Step[0050/127] Total:3.7003 Init:2.8550 Final:0.8453
2026-06-08 16:16:25.529262 Ep[001/70] Step[0100/127] Total:3.5804 Init:2.7687 Final:0.8118
2026-06-08 16:17:54.115640 Ep[001/70] Step[0127/127] Total:2.9757 Init:2.3237 Final:0.6520
Ep:1  MAE:0.111920  bestMAE:1.000000  bestEp:0
  -> New best! Saved epoch 1
[Ep   1/70] LR=2.00e-04 Loss=3.8024 MAE=0.111920
2026-06-08 16:18:12.862551 Ep[002/70] Step[0001/127] Total:3.1746 Init:2.4656 Final:0.7090
2026-06-08 16:20:32.937629 Ep[002/70] Step[0050/127] Total:3.3677 Init:2.5779 Final:0.7898
2026-06-08 16:22:55.552892 Ep[002/70] Step[0100/127] Total:2.8895 Init:2.2272 Final:0.6623
2026-06-08 16:24:10.318916 Ep[002/70] Step[0127/127] Total:2.5000 Init:1.9729 Final:0.5271
Ep:2  MAE:0.095118  bestMAE:0.111920  bestEp:1
  -> New best! Saved epoch 2
[Ep   2/70] LR=3.50e-04 Lo

## Bước 9 — Vẽ biểu đồ Loss và MAE

In [9]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend, tránh lỗi display
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os

CSV_PATH = '/kaggle/working/SINet-V2/snapshot/SINet_V2_PVTv2/training_log.csv'
PLOT_DIR = '/kaggle/working/plots/'
os.makedirs(PLOT_DIR, exist_ok=True)

assert os.path.exists(CSV_PATH), f'Chưa có log: {CSV_PATH}'
df = pd.read_csv(CSV_PATH)

# Ép kiểu an toàn
for col in ['epoch','best_epoch']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
for col in ['lr','loss_total','loss_init','loss_final','mae','best_mae']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Tổng số epoch đã train: {len(df)}')
print(df.to_string(index=False))

best_row   = df.loc[df['mae'].idxmin()]
best_ep    = int(best_row['epoch'])
best_mae_v = float(best_row['mae'])
print(f'\n★ Best MAE = {best_mae_v:.6f}  tại Epoch {best_ep}')

epochs = df['epoch'].values

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'legend.fontsize': 11, 'lines.linewidth': 2,
    'axes.grid': True, 'grid.alpha': 0.35,
})

# ── Biểu đồ 1: Loss ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, df['loss_total'], color='#E63946', label='Total loss',   lw=2.5)
ax.plot(epochs, df['loss_init'],  color='#457B9D', label='Init loss',    lw=1.8, ls='--')
ax.plot(epochs, df['loss_final'], color='#2A9D8F', label='Final loss',   lw=1.8, ls=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('SINet-V2 PVT-v2 — Training Loss')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
p1 = PLOT_DIR + 'loss_curve.png'
plt.savefig(p1, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ {p1}')

# ── Biểu đồ 2: MAE ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, df['mae'],      color='#E76F51', label='Val MAE (CAMO)', lw=2.5)
ax.plot(epochs, df['best_mae'], color='#2A9D8F', label='Best MAE so far',lw=1.5, ls='--', alpha=0.8)
ax.scatter([best_ep], [best_mae_v], color='red', zorder=5, s=80)
ax.annotate(
    f'Best\nEp {best_ep}\n{best_mae_v:.4f}',
    xy=(best_ep, best_mae_v),
    xytext=(best_ep + max(1, len(df)//10), best_mae_v + 0.003),
    fontsize=10, color='red',
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
ax.set_title('SINet-V2 PVT-v2 — Validation MAE (CAMO)')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
p2 = PLOT_DIR + 'mae_curve.png'
plt.savefig(p2, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ {p2}')

# ── Biểu đồ 3: Loss + MAE dual axis ─────────────────────────
fig, ax1 = plt.subplots(figsize=(12, 5))
c_loss = '#E63946'; c_mae = '#457B9D'
ax1.plot(epochs, df['loss_total'], color=c_loss, lw=2.5, label='Total loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color=c_loss)
ax1.tick_params(axis='y', labelcolor=c_loss)
ax2 = ax1.twinx()
ax2.plot(epochs, df['mae'], color=c_mae, lw=2.5, ls='--', label='Val MAE')
ax2.set_ylabel('MAE', color=c_mae)
ax2.tick_params(axis='y', labelcolor=c_mae)
ax2.scatter([best_ep], [best_mae_v], color='red', zorder=5, s=80)
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper right')
ax1.set_title('SINet-V2 PVT-v2 — Loss & MAE')
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax1.grid(True, alpha=0.3)
plt.tight_layout()
p3 = PLOT_DIR + 'loss_mae_combined.png'
plt.savefig(p3, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ {p3}')

# ── Biểu đồ 4: LR schedule ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
lr_vals = df['lr'].values.astype(float)
ax.plot(epochs, lr_vals, color='#6A4C93', lw=2.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning rate')
ax.set_title('Cosine LR schedule (head)')
ax.set_yscale('log')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
p4 = PLOT_DIR + 'lr_schedule.png'
plt.savefig(p4, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'✅ {p4}')

print(f'\n✅ Tất cả biểu đồ lưu tại {PLOT_DIR}')

Tổng số epoch đã train: 70
 epoch       lr  loss_total  loss_init  loss_final      mae  best_mae  best_epoch
     1 0.000200      3.8024     2.9505      0.8519 0.111920  0.111920           1
     2 0.000350      2.9869     2.3093      0.6776 0.095118  0.095118           2
     3 0.000500      2.6725     2.0539      0.6187 0.082763  0.082763           3
     4 0.000500      2.4679     1.8949      0.5730 0.074056  0.074056           4
     5 0.000499      2.3399     1.7988      0.5411 0.069817  0.069817           5
     6 0.000498      2.2362     1.7221      0.5141 0.073133  0.069817           5
     7 0.000496      2.1773     1.6784      0.4990 0.069544  0.069544           7
     8 0.000493      2.1067     1.6275      0.4792 0.067215  0.067215           8
     9 0.000490      2.0468     1.5863      0.4605 0.060668  0.060668           9
    10 0.000487      2.0146     1.5633      0.4513 0.066713  0.060668           9
    11 0.000483      1.9696     1.5312      0.4383 0.057029  0.057029  

## Bước 10 — Inference trên 3 bộ test

In [10]:
import imageio, torch, torch.nn.functional as F
import numpy as np, os, sys

WORK = '/kaggle/working/SINet-V2'
sys.path.insert(0, WORK)
os.chdir(WORK)

from lib.Network_PVTv2_GRA_NCD import Network
from utils.data_val import test_dataset

MODEL_PATH = './snapshot/SINet_V2_PVTv2/Net_epoch_best.pth'
assert os.path.exists(MODEL_PATH), f'Không tìm thấy: {MODEL_PATH}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Model: {MODEL_PATH}')
print(f'Device: {device}')

mae_results = {}
for dataset in ['CAMO', 'COD10K', 'CHAMELEON']:
    img_dir = f'./Dataset/TestDataset/{dataset}/Imgs/'
    gt_dir  = f'./Dataset/TestDataset/{dataset}/GT/'
    if not os.path.exists(img_dir):
        print(f'⚠️  Bỏ qua {dataset} (không có data)')
        continue
    save_path = f'./res/SINet_V2_PVTv2/{dataset}/'
    os.makedirs(save_path, exist_ok=True)

    model = Network(channel=32, pvt_variant='b2', imagenet_pretrained=False)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device).eval()

    loader = test_dataset(img_dir, gt_dir, 352)
    mae_sum = 0
    for _ in range(loader.size):
        image, gt, name, _ = loader.load_data()
        gt = np.asarray(gt, np.float32)
        gt /= (gt.max() + 1e-8)
        with torch.no_grad():
            res = model(image.to(device))
        res = F.interpolate(res[3], size=gt.shape, mode='bilinear', align_corners=False)
        res = res.sigmoid().cpu().numpy().squeeze()
        res = (res - res.min()) / (res.max() - res.min() + 1e-8)
        mae_sum += np.abs(res - gt).sum() / (gt.shape[0] * gt.shape[1])
        imageio.imwrite(save_path + name, (res*255).astype(np.uint8))

    mae_results[dataset] = mae_sum / loader.size
    del model
    torch.cuda.empty_cache()
    print(f'✅ {dataset:10s}  {loader.size} ảnh  MAE = {mae_results[dataset]:.6f}')

print('\n=== Final MAE ===')
for ds, mae in mae_results.items():
    print(f'  {ds:10s}: {mae:.6f}')

Model: ./snapshot/SINet_V2_PVTv2/Net_epoch_best.pth
Device: cuda
✅ CAMO        250 ảnh  MAE = 0.050882
✅ COD10K      2026 ảnh  MAE = 0.028109
✅ CHAMELEON   76 ảnh  MAE = 0.026300

=== Final MAE ===
  CAMO      : 0.050882
  COD10K    : 0.028109
  CHAMELEON : 0.026300


## Bước 11 — Lưu training summary (JSON)

In [11]:
import json, pandas as pd, numpy as np, os

CSV_PATH = '/kaggle/working/SINet-V2/snapshot/SINet_V2_PVTv2/training_log.csv'
df = pd.read_csv(CSV_PATH)
for col in ['epoch','best_epoch']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
for col in ['lr','loss_total','loss_init','loss_final','mae','best_mae']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

best_row = df.loc[df['mae'].idxmin()]

summary = {
    'model':   'SINet-V2 (PVT-v2-B2 backbone)',
    'hyperparameters': {
        'total_epochs':  25,
        'lr_head':       5e-4,
        'lr_backbone':   5e-5,
        'lr_schedule':   'cosine annealing + 3-epoch warmup',
        'batch_size':    32,
        'input_size':    352,
        'optimizer':     'Adam',
        'clip_gradient': 0.5,
        'loss':          'Weighted BCE + Weighted IoU',
    },
    'datasets': {
        'train': 'COD10K-train + CAMO-train (4040 images)',
        'val':   'CAMO test set',
    },
    'results': {
        'best_epoch':    int(best_row['epoch']),
        'best_mae_camo': float(best_row['mae']),
        'final_loss':    float(df.iloc[-1]['loss_total']),
        'mae_per_dataset': ({k: float(v) for k, v in mae_results.items()}
                            if 'mae_results' in dir() else {}),
    },
}

out = '/kaggle/working/training_summary.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('✅', out)
print(json.dumps(summary, indent=2, ensure_ascii=False))

✅ /kaggle/working/training_summary.json
{
  "model": "SINet-V2 (PVT-v2-B2 backbone)",
  "hyperparameters": {
    "total_epochs": 25,
    "lr_head": 0.0005,
    "lr_backbone": 5e-05,
    "lr_schedule": "cosine annealing + 3-epoch warmup",
    "batch_size": 32,
    "input_size": 352,
    "optimizer": "Adam",
    "clip_gradient": 0.5,
    "loss": "Weighted BCE + Weighted IoU"
  },
  "datasets": {
    "train": "COD10K-train + CAMO-train (4040 images)",
    "val": "CAMO test set"
  },
  "results": {
    "best_epoch": 60,
    "best_mae_camo": 0.050882,
    "final_loss": 1.4582,
    "mae_per_dataset": {
      "CAMO": 0.0508822537958622,
      "COD10K": 0.02810928411781788,
      "CHAMELEON": 0.02630019746720791
    }
  }
}


## Bước 12 — Zip kết quả để download

In [12]:
import shutil, os

WORK = '/kaggle/working/SINet-V2'
OUT  = '/kaggle/working/SINet_V2_Results/'
os.makedirs(OUT + 'checkpoints', exist_ok=True)
os.makedirs(OUT + 'plots',       exist_ok=True)

# Checkpoints
SNAP = f'{WORK}/snapshot/SINet_V2_PVTv2/'
if os.path.exists(SNAP):
    shutil.copytree(SNAP, OUT + 'checkpoints', dirs_exist_ok=True)
    print(f'✅ Checkpoints: {len(os.listdir(OUT+"checkpoints"))} files')

# Plots
if os.path.exists('/kaggle/working/plots'):
    shutil.copytree('/kaggle/working/plots', OUT + 'plots', dirs_exist_ok=True)

# Log + JSON
for src in [SNAP + 'training_log.csv',
            '/kaggle/working/training_summary.json']:
    if os.path.exists(src):
        shutil.copy(src, OUT)

# Inference results
res_src = f'{WORK}/res/SINet_V2_PVTv2/'
if os.path.exists(res_src):
    shutil.copytree(res_src, OUT + 'res', dirs_exist_ok=True)

print('⏳ Đang zip...')
shutil.make_archive('/kaggle/working/SINet_V2_Results', 'zip',
                    '/kaggle/working/SINet_V2_Results/')
size = os.path.getsize('/kaggle/working/SINet_V2_Results.zip') / (1024**2)
print(f'\n✅ SINet_V2_Results.zip  ({size:.1f} MB)')
print('👉 Tab OUTPUT bên phải → download SINet_V2_Results.zip')

✅ Checkpoints: 17 files
⏳ Đang zip...

✅ SINet_V2_Results.zip  (1425.5 MB)
👉 Tab OUTPUT bên phải → download SINet_V2_Results.zip
